In [1]:
# -*- coding: utf-8 -*-
from heapq import heappush, heappop

class Node:
    def __init__(self, current_city, visited, g, h, parent=None):
        self.current_city = current_city
        self.visited = visited  # Tuple chứa các thành phố đã qua
        self.g = g              # Chi phí thực tế đã đi
        self.h = h              # Chi phí ước lượng
        self.f = g + h          # f(n) = g(n) + h(n)
        self.parent = parent

    def __lt__(self, other):
        # Ưu tiên f(n) nhỏ nhất, nếu f bằng nhau thì ưu tiên h(n) nhỏ hơn
        if self.f == other.f:
            return self.h < other.h
        return self.f < other.f

def calculate_h(current_city, visited, matrix, num_cities):
    """
    Hàm Heuristic: Ước lượng chi phí rẻ nhất để đi qua các thành phố còn lại
    và quay về điểm xuất phát.
    """
    not_visited = [i for i in range(num_cities) if i not in visited]

    if not not_visited:
        return matrix[current_city][0] # Quay về điểm đầu

    # h = Cạnh nhỏ nhất rời khỏi thành phố hiện tại + Cạnh nhỏ nhất để quay về 0
    # Đây là một Heuristic đơn giản nhưng đảm bảo tính đúng đắn cho TSP
    min_to_next = min([matrix[current_city][i] for i in not_visited])
    min_back_home = min([matrix[i][0] for i in not_visited])

    return min_to_next + min_back_home

def solve_tsp_akt(matrix):
    num_cities = len(matrix)
    start_city = 0

    # Khởi tạo
    h_init = calculate_h(start_city, (start_city,), matrix, num_cities)
    root = Node(start_city, (start_city,), 0, h_init)

    pq = []
    heappush(pq, root)

    visited_nodes_count = 0

    print("\n" + "="*50)
    print("BẮT ĐẦU GIẢI BÀI TOÁN NGƯỜI GIAO HÀNG (AKT/A*)")
    print("="*50)

    while pq:
        node = heappop(pq)
        visited_nodes_count += 1

        # In thông tin node đang xét
        path_str = " -> ".join(map(str, node.visited))
        print(f"Duyệt Node: [{path_str}]")
        print(f"  -> g(n)={node.g}, h(n)={node.h}, f(n)={node.f}")
        print("-" * 30)

        # Điều kiện đích: Đã thăm hết và quay về 0
        if len(node.visited) == num_cities:
            final_cost = node.g + matrix[node.current_city][0]
            final_path = list(node.visited) + [0]

            print("\n" + "*"*20 + " KẾT QUẢ " + "*"*20)
            print(f"Đường đi tối ưu: {' -> '.join(map(str, final_path))}")
            print(f"Tổng chi phí tối thiểu (f): {final_cost}")
            print(f"Số trạng thái đã xét: {visited_nodes_count}")
            return

        # Sinh các trạng thái kế tiếp (Neighbors)
        for next_city in range(num_cities):
            if next_city not in node.visited:
                g_next = node.g + matrix[node.current_city][next_city]
                visited_next = node.visited + (next_city,)
                h_next = calculate_h(next_city, visited_next, matrix, num_cities)

                child = Node(next_city, visited_next, g_next, h_next, node)
                heappush(pq, child)

# --- THIẾT LẬP DỮ LIỆU ---
if __name__ == "__main__":
    # Ma trận chi phí mặc định (Trọng số giữa 4 thành phố 0, 1, 2, 3)
    # Ví dụ: dist[0][1] = 10 là chi phí đi từ thành phố 0 đến 1
    dist_matrix = [
        [0, 10, 15, 20],
        [10, 0, 35, 25],
        [15, 35, 0, 30],
        [20, 25, 30, 0]
    ]

    print("GIÁ TRỊ MẶC ĐỊNH (Ma trận chi phí giữa các thành phố):")
    for row in dist_matrix:
        print("  ", row)

    solve_tsp_akt(dist_matrix)

GIÁ TRỊ MẶC ĐỊNH (Ma trận chi phí giữa các thành phố):
   [0, 10, 15, 20]
   [10, 0, 35, 25]
   [15, 35, 0, 30]
   [20, 25, 30, 0]

BẮT ĐẦU GIẢI BÀI TOÁN NGƯỜI GIAO HÀNG (AKT/A*)
Duyệt Node: [0]
  -> g(n)=0, h(n)=20, f(n)=20
------------------------------
Duyệt Node: [0 -> 1]
  -> g(n)=10, h(n)=40, f(n)=50
------------------------------
Duyệt Node: [0 -> 3]
  -> g(n)=20, h(n)=35, f(n)=55
------------------------------
Duyệt Node: [0 -> 2]
  -> g(n)=15, h(n)=40, f(n)=55
------------------------------
Duyệt Node: [0 -> 2 -> 3]
  -> g(n)=45, h(n)=35, f(n)=80
------------------------------
Duyệt Node: [0 -> 2 -> 3 -> 1]
  -> g(n)=70, h(n)=10, f(n)=80
------------------------------

******************** KẾT QUẢ ********************
Đường đi tối ưu: 0 -> 2 -> 3 -> 1 -> 0
Tổng chi phí tối thiểu (f): 80
Số trạng thái đã xét: 6
